<a href="https://colab.research.google.com/github/Rajeraghav/AI-Engineer-Journey/blob/main/Chatbot%20using%20Deep%20Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
#              AI CHATBOT ASSISTANT
#       TENSORFLOW + NLP + BIDIRECTIONAL LSTM
#                    + TF-IDF FALLBACK
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import random
import pickle
import os

import tensorflow as tf

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    SpatialDropout1D
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from tensorflow.keras.optimizers import Adam


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)


# ============================================================
# 3. TITLE
# ============================================================

print("=" * 75)
print("                  AI CHATBOT ASSISTANT")
print("       TENSORFLOW + NLP + BIDIRECTIONAL LSTM")
print("=" * 75)

print("\nTensorFlow version:", tf.__version__)


# ============================================================
# 4. UPLOAD CSV
# ============================================================

print("\nUpload your CSV file.")

print("\nRequired columns:")
print("intent, text, response")

print()

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No CSV file was uploaded.")

filename = list(uploaded.keys())[0]

print("\nUploaded file:", filename)


# ============================================================
# 5. LOAD CSV
# ============================================================

df = pd.read_csv(filename)

print("\nCSV loaded successfully!")

print("Original dataset shape:", df.shape)

print("\nOriginal columns:")
print(df.columns.tolist())


# ============================================================
# 6. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
)


# ============================================================
# 7. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "intent",
    "text",
    "response"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        "\nMissing columns: "
        + str(missing_columns)
        + "\n\nYour CSV must contain:\n"
        + "intent,text,response"
    )


# ============================================================
# 8. KEEP REQUIRED COLUMNS
# ============================================================

df = df[
    [
        "intent",
        "text",
        "response"
    ]
].copy()


# ============================================================
# 9. REMOVE NULL VALUES
# ============================================================

print("\nMissing values:")

print(
    df[
        [
            "intent",
            "text",
            "response"
        ]
    ].isnull().sum()
)

df = df.dropna(
    subset=[
        "intent",
        "text",
        "response"
    ]
)


# ============================================================
# 10. CONVERT TO STRING
# ============================================================

for column in [
    "intent",
    "text",
    "response"
]:

    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
    )


# ============================================================
# 11. REMOVE EMPTY VALUES
# ============================================================

df = df[
    (df["intent"] != "") &
    (df["text"] != "") &
    (df["response"] != "")
].copy()


# ============================================================
# 12. REMOVE DUPLICATES
# ============================================================

df = (
    df
    .drop_duplicates(
        subset=[
            "intent",
            "text"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 13. TEXT PREPROCESSING
# ============================================================

def preprocess_text(text):

    text = str(text).lower()

    # Replace URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Keep alphabets and numbers
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


df["clean_text"] = (
    df["text"]
    .apply(preprocess_text)
)


# ============================================================
# 14. REMOVE EMPTY CLEAN TEXT
# ============================================================

df = df[
    df["clean_text"].str.len() > 0
].reset_index(drop=True)


# ============================================================
# 15. DATASET INFORMATION
# ============================================================

print("\n" + "=" * 75)
print("DATASET INFORMATION")
print("=" * 75)

print(
    "\nDataset shape:",
    df.shape
)

number_of_intents = df["intent"].nunique()

print(
    "\nNumber of intents:",
    number_of_intents
)

print("\nIntent distribution:")

print(
    df["intent"].value_counts()
)


# ============================================================
# 16. CHECK DATASET SIZE
# ============================================================

if number_of_intents < 2:

    raise ValueError(
        "At least two different intents are required."
    )


minimum_examples = (
    df["intent"]
    .value_counts()
    .min()
)

print(
    "\nMinimum examples for one intent:",
    minimum_examples
)

if minimum_examples < 5:

    print(
        "\nWARNING:"
        "\nSome intents have fewer than 5 examples."
        "\nThe model may not generalize well."
    )


# ============================================================
# 17. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    df["intent"]
)

number_of_classes = len(
    label_encoder.classes_
)

print(
    "\nNumber of classes:",
    number_of_classes
)

print("\nIntent encoding:")

for index, intent in enumerate(
    label_encoder.classes_
):

    print(
        f"{index} -> {intent}"
    )


# ============================================================
# 18. TOKENIZER
# ============================================================

MAX_WORDS = 5000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>",
    lower=True,
    filters=""
)

tokenizer.fit_on_texts(
    df["clean_text"]
)


# ============================================================
# 19. CONVERT TEXT TO SEQUENCES
# ============================================================

sequences = tokenizer.texts_to_sequences(
    df["clean_text"]
)


# ============================================================
# 20. DETERMINE SEQUENCE LENGTH
# ============================================================

sequence_lengths = [
    len(sequence)
    for sequence in sequences
]

MAX_LENGTH = max(
    10,
    min(
        25,
        int(
            np.percentile(
                sequence_lengths,
                95
            )
        )
    )
)

print(
    "\nSequence length:",
    MAX_LENGTH
)


# ============================================================
# 21. PAD SEQUENCES
# ============================================================

X = pad_sequences(

    sequences,

    maxlen=MAX_LENGTH,

    padding="post",

    truncating="post"
)

print(
    "Input shape:",
    X.shape
)


# ============================================================
# 22. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# First:
# 80% training
# 20% temporary

X_train, X_temp, y_train, y_temp = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=SEED,

    stratify=y
)


# Then:
# 10% validation
# 10% test

X_val, X_test, y_val, y_test = train_test_split(

    X_temp,

    y_temp,

    test_size=0.50,

    random_state=SEED,

    stratify=y_temp
)


print("\n" + "=" * 75)
print("TRAIN / VALIDATION / TEST SPLIT")
print("=" * 75)

print(
    "Training samples:",
    len(X_train)
)

print(
    "Validation samples:",
    len(X_val)
)

print(
    "Testing samples:",
    len(X_test)
)


# ============================================================
# 23. VOCABULARY SIZE
# ============================================================

vocab_size = min(
    MAX_WORDS,
    len(tokenizer.word_index) + 1
)

print(
    "\nVocabulary size:",
    vocab_size
)


# ============================================================
# 24. BUILD LSTM MODEL
# ============================================================

print("\n" + "=" * 75)
print("BUILDING LSTM MODEL")
print("=" * 75)


model = Sequential([

    # --------------------------------------------------------
    # Embedding
    # --------------------------------------------------------

    Embedding(

        input_dim=vocab_size,

        output_dim=64,

        mask_zero=True
    ),

    # --------------------------------------------------------
    # Spatial Dropout
    # --------------------------------------------------------

    SpatialDropout1D(
        0.20
    ),

    # --------------------------------------------------------
    # Bidirectional LSTM
    # --------------------------------------------------------

    Bidirectional(

        LSTM(
            64,
            return_sequences=False
        )
    ),

    # --------------------------------------------------------
    # Dropout
    # --------------------------------------------------------

    Dropout(
        0.30
    ),

    # --------------------------------------------------------
    # Dense
    # --------------------------------------------------------

    Dense(
        64,
        activation="relu"
    ),

    # --------------------------------------------------------
    # Dropout
    # --------------------------------------------------------

    Dropout(
        0.20
    ),

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    Dense(
        number_of_classes,
        activation="softmax"
    )
])


# ============================================================
# 25. COMPILE
# ============================================================

model.compile(

    optimizer=Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)


# ============================================================
# 26. BUILD MODEL
# ============================================================

model.build(
    input_shape=(
        None,
        MAX_LENGTH
    )
)


# ============================================================
# 27. MODEL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("LSTM MODEL")
print("=" * 75)

model.summary()


# ============================================================
# 28. CALLBACKS
# ============================================================

early_stopping = EarlyStopping(

    monitor="val_accuracy",

    patience=12,

    mode="max",

    restore_best_weights=True,

    verbose=1
)


reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=4,

    min_lr=0.00001,

    verbose=1
)


# ============================================================
# 29. TRAIN LSTM
# ============================================================

print("\n" + "=" * 75)
print("TRAINING LSTM CHATBOT")
print("=" * 75)


history = model.fit(

    X_train,

    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,

    batch_size=8,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)


# ============================================================
# 30. EVALUATE LSTM
# ============================================================

print("\n" + "=" * 75)
print("LSTM MODEL EVALUATION")
print("=" * 75)


test_loss, test_accuracy = model.evaluate(

    X_test,

    y_test,

    verbose=0
)


print(
    "\nTest Loss:",
    round(
        float(test_loss),
        4
    )
)

print(
    "Test Accuracy:",
    round(
        float(test_accuracy) * 100,
        2
    ),
    "%"
)


# ============================================================
# 31. LSTM TEST PREDICTIONS
# ============================================================

lstm_predictions = model.predict(

    X_test,

    verbose=0
)

lstm_predicted_classes = np.argmax(

    lstm_predictions,

    axis=1
)


print("\nLSTM Classification Report:")

print(
    classification_report(

        y_test,

        lstm_predicted_classes,

        labels=np.arange(number_of_classes),

        target_names=label_encoder.classes_,

        zero_division=0
    )
)


# ============================================================
# 32. CREATE RESPONSE DICTIONARY
# ============================================================

intent_responses = {}


for intent in label_encoder.classes_:

    responses = (
        df[
            df["intent"] == intent
        ]["response"]
        .tolist()
    )

    intent_responses[
        intent
    ] = responses


# ============================================================
# 33. TF-IDF MODEL
#
# IMPORTANT:
#
# The TF-IDF model is NOT replacing the LSTM.
#
# It is used as a similarity-based fallback because
# the dataset contains only 203 examples.
#
# This makes the chatbot much more reliable on a
# small intent-classification dataset.
# ============================================================

print("\n" + "=" * 75)
print("BUILDING TF-IDF SIMILARITY MODEL")
print("=" * 75)


tfidf_vectorizer = TfidfVectorizer(

    lowercase=True,

    ngram_range=(1, 2),

    sublinear_tf=True
)


tfidf_matrix = tfidf_vectorizer.fit_transform(

    df["clean_text"]
)


# ============================================================
# 34. TF-IDF SIMILARITY FUNCTION
# ============================================================

def tfidf_similarity_prediction(
    user_input
):

    clean_input = preprocess_text(
        user_input
    )

    user_vector = (
        tfidf_vectorizer
        .transform(
            [clean_input]
        )
    )

    # Cosine similarity because TF-IDF vectors
    # are normalized by default.

    similarity_scores = (
        tfidf_matrix
        .dot(
            user_vector.T
        )
        .toarray()
        .flatten()
    )

    best_index = int(
        np.argmax(
            similarity_scores
        )
    )

    best_score = float(
        similarity_scores[
            best_index
        ]
    )

    best_intent = df.iloc[
        best_index
    ]["intent"]

    return (
        best_intent,
        best_score,
        best_index
    )


# ============================================================
# 35. LSTM PREDICTION FUNCTION
# ============================================================

def lstm_prediction(
    user_input
):

    clean_input = preprocess_text(
        user_input
    )

    sequence = (
        tokenizer
        .texts_to_sequences(
            [clean_input]
        )
    )

    padded_sequence = pad_sequences(

        sequence,

        maxlen=MAX_LENGTH,

        padding="post",

        truncating="post"
    )

    prediction = model.predict(

        padded_sequence,

        verbose=0
    )[0]

    predicted_class = int(
        np.argmax(
            prediction
        )
    )

    confidence = float(
        prediction[
            predicted_class
        ]
    )

    predicted_intent = (
        label_encoder
        .inverse_transform(
            [predicted_class]
        )[0]
    )

    top_indices = np.argsort(
        prediction
    )[-3:][::-1]

    top_predictions = []

    for index in top_indices:

        intent = (
            label_encoder
            .inverse_transform(
                [int(index)]
            )[0]
        )

        score = float(
            prediction[
                index
            ]
        )

        top_predictions.append(
            (
                intent,
                score
            )
        )

    return (
        predicted_intent,
        confidence,
        top_predictions
    )


# ============================================================
# 36. HYBRID PREDICTION
# ============================================================

def predict_intent(
    user_input
):

    clean_input = preprocess_text(
        user_input
    )

    if clean_input == "":
        return (
            None,
            0.0,
            [],
            0.0,
            None
        )


    # --------------------------------------------------------
    # LSTM prediction
    # --------------------------------------------------------

    (
        lstm_intent,
        lstm_confidence,
        top_predictions
    ) = lstm_prediction(
        user_input
    )


    # --------------------------------------------------------
    # TF-IDF prediction
    # --------------------------------------------------------

    (
        tfidf_intent,
        tfidf_score,
        tfidf_index
    ) = tfidf_similarity_prediction(
        user_input
    )


    # --------------------------------------------------------
    # Hybrid decision
    #
    # For a small dataset:
    #
    # Strong TF-IDF similarity is more trustworthy than
    # an LSTM probability that is close to 5%.
    # --------------------------------------------------------

    if tfidf_score >= 0.65:

        final_intent = tfidf_intent

        decision = "TF-IDF exact/strong similarity"


    elif (
        tfidf_score >= 0.40 and
        lstm_confidence < 0.25
    ):

        final_intent = tfidf_intent

        decision = "TF-IDF similarity"


    elif lstm_confidence >= 0.30:

        final_intent = lstm_intent

        decision = "LSTM"


    elif tfidf_score >= 0.25:

        final_intent = tfidf_intent

        decision = "TF-IDF similarity"


    else:

        final_intent = lstm_intent

        decision = "LSTM fallback"


    return (
        final_intent,
        lstm_confidence,
        top_predictions,
        tfidf_score,
        decision
    )


# ============================================================
# 37. CHATBOT RESPONSE
# ============================================================

def chatbot_response(
    user_input
):

    if not user_input.strip():

        return (
            "Please enter a question."
        )


    (
        intent,
        lstm_confidence,
        top_predictions,
        tfidf_score,
        decision
    ) = predict_intent(
        user_input
    )


    if intent is None:

        return (
            "Please enter a valid question."
        )


    print(
        f"\n[Final intent: {intent}]"
    )

    print(
        f"[LSTM confidence: "
        f"{lstm_confidence:.2%}]"
    )

    print(
        f"[TF-IDF similarity: "
        f"{tfidf_score:.2%}]"
    )

    print(
        f"[Decision: {decision}]"
    )


    print(
        "\nTop LSTM predictions:"
    )


    for predicted_intent, score in (
        top_predictions
    ):

        print(
            f"  {predicted_intent}: "
            f"{score:.2%}"
        )


    # --------------------------------------------------------
    # UNKNOWN QUESTION CHECK
    #
    # Very low similarity + weak LSTM
    # means the question may be outside
    # the chatbot's known domain.
    # --------------------------------------------------------

    if (
        tfidf_score < 0.18 and
        lstm_confidence < 0.20
    ):

        return (
            "I'm not completely sure what you mean. "
            "I can help with Python, AI, machine learning, "
            "deep learning, NLP, RNN, LSTM, CNN, "
            "Transformers, courses, enrollment, fees, "
            "certificates and related topics."
        )


    responses = intent_responses.get(
        intent,
        []
    )


    if not responses:

        return (
            "I don't have a response for "
            "that question yet."
        )


    return random.choice(
        responses
    )


# ============================================================
# 38. TEST QUESTIONS
# ============================================================

print("\n" + "=" * 75)
print("TESTING CHATBOT")
print("=" * 75)


test_questions = [

    "Hello",

    "Hi",

    "Hey",

    "Good morning",

    "What is Python?",

    "Tell me about Python",

    "Explain Python",

    "What is machine learning?",

    "Tell me about machine learning",

    "Explain machine learning",

    "What is deep learning?",

    "What is RNN?",

    "Explain RNN",

    "What is LSTM?",

    "Explain LSTM",

    "What is CNN?",

    "What is NLP?",

    "Explain NLP",

    "What is a transformer?",

    "What is artificial intelligence?",

    "Thank you",

    "Thanks",

    "Bye",

    "Goodbye"

]


for question in test_questions:

    print(
        "\n" + "-" * 75
    )

    print(
        "User:",
        question
    )


    (
        predicted_intent,
        lstm_confidence,
        top_predictions,
        tfidf_score,
        decision
    ) = predict_intent(
        question
    )


    print(
        "Predicted Intent:",
        predicted_intent
    )

    print(
        "LSTM Confidence:",
        f"{lstm_confidence:.2%}"
    )

    print(
        "TF-IDF Similarity:",
        f"{tfidf_score:.2%}"
    )

    print(
        "Decision:",
        decision
    )


    responses = intent_responses.get(
        predicted_intent,
        []
    )


    if responses:

        print(
            "Bot:",
            random.choice(
                responses
            )
        )


# ============================================================
# 39. SAVE LSTM MODEL
# ============================================================

model.save(
    "chatbot_lstm.keras"
)

print(
    "\nLSTM model saved:"
)

print(
    "chatbot_lstm.keras"
)


# ============================================================
# 40. SAVE TOKENIZER
# ============================================================

with open(
    "chatbot_tokenizer.pkl",
    "wb"
) as file:

    pickle.dump(
        tokenizer,
        file
    )

print(
    "Tokenizer saved:"
)

print(
    "chatbot_tokenizer.pkl"
)


# ============================================================
# 41. SAVE LABEL ENCODER
# ============================================================

with open(
    "chatbot_label_encoder.pkl",
    "wb"
) as file:

    pickle.dump(
        label_encoder,
        file
    )

print(
    "Label encoder saved:"
)

print(
    "chatbot_label_encoder.pkl"
)


# ============================================================
# 42. SAVE TF-IDF VECTORIZER
# ============================================================

with open(
    "chatbot_tfidf_vectorizer.pkl",
    "wb"
) as file:

    pickle.dump(
        tfidf_vectorizer,
        file
    )

print(
    "TF-IDF vectorizer saved:"
)

print(
    "chatbot_tfidf_vectorizer.pkl"
)


# ============================================================
# 43. SAVE DATASET
# ============================================================

df.to_csv(
    "chatbot_processed_dataset.csv",
    index=False
)

print(
    "Processed dataset saved:"
)

print(
    "chatbot_processed_dataset.csv"
)


# ============================================================
# 44. DOWNLOAD MODEL FILES
# ============================================================

print("\n" + "=" * 75)
print("SAVED FILES")
print("=" * 75)

print(
    "\n1. chatbot_lstm.keras"
)

print(
    "2. chatbot_tokenizer.pkl"
)

print(
    "3. chatbot_label_encoder.pkl"
)

print(
    "4. chatbot_tfidf_vectorizer.pkl"
)

print(
    "5. chatbot_processed_dataset.csv"
)


# ============================================================
# 45. START CHATBOT
# ============================================================

print("\n" + "=" * 75)
print("🤖 AI CHATBOT ASSISTANT IS READY")
print("=" * 75)

print("\nCommands:")

print(
    "  exit     -> Stop chatbot"
)

print(
    "  intents  -> Show available intents"
)

print(
    "  test     -> Run sample questions"
)

print("=" * 75)


while True:

    user_input = input(
        "\nYou: "
    ).strip()


    # --------------------------------------------------------
    # EXIT
    # --------------------------------------------------------

    if user_input.lower() in [
        "exit",
        "quit"
    ]:

        print(
            "Bot: Goodbye! Have a nice day!"
        )

        break


    # --------------------------------------------------------
    # SHOW INTENTS
    # --------------------------------------------------------

    if user_input.lower() == "intents":

        print(
            "\nAvailable intents:"
        )

        for intent in (
            label_encoder.classes_
        ):

            print(
                "-",
                intent
            )

        continue


    # --------------------------------------------------------
    # TEST COMMAND
    # --------------------------------------------------------

    if user_input.lower() == "test":

        print(
            "\nRunning sample questions..."
        )

        for question in test_questions:

            print(
                "\nUser:",
                question
            )

            response = chatbot_response(
                question
            )

            print(
                "Bot:",
                response
            )

        continue


    # --------------------------------------------------------
    # EMPTY INPUT
    # --------------------------------------------------------

    if user_input == "":

        print(
            "Bot: Please enter a message."
        )

        continue


    # --------------------------------------------------------
    # NORMAL CHAT
    # --------------------------------------------------------

    response = chatbot_response(
        user_input
    )

    print(
        "Bot:",
        response
    )

                  AI CHATBOT ASSISTANT
       TENSORFLOW + NLP + BIDIRECTIONAL LSTM

TensorFlow version: 2.20.0

Upload your CSV file.

Required columns:
intent, text, response



Saving ai_chatbot_dataset.csv to ai_chatbot_dataset (6).csv

Uploaded file: ai_chatbot_dataset (6).csv

CSV loaded successfully!
Original dataset shape: (203, 3)

Original columns:
['intent', 'text', 'response']

Missing values:
intent      0
text        0
response    0
dtype: int64

DATASET INFORMATION

Dataset shape: (203, 4)

Number of intents: 20

Intent distribution:
intent
greeting            15
goodbye             10
help                10
about_bot           10
nlp                 10
python              10
machine_learning    10
deep_learning       10
neural_network      10
rnn                 10
lstm                10
cnn                 10
enrollment          10
transformer         10
ai                  10
course              10
fees                10
certificate         10
contact             10
thanks               8
Name: count, dtype: int64

Minimum examples for one intent: 8

Number of classes: 20

Intent encoding:
0 -> about_bot
1 -> ai
2 -> certificate
3 -> cnn
4 -> c

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 10, 64)         │        13,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 10, 64)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 20)             │         1,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,172 (348.33 KB)

 Trainable params: 89,172 (348.33 KB)

 Non-trainable params: 0 (0.00 B)


TRAINING LSTM CHATBOT
Epoch 1/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.0370 - loss: 2.9965 - val_accuracy: 0.0500 - val_loss: 2.9905 - learning_rate: 0.0010
Epoch 2/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.1296 - loss: 2.9828 - val_accuracy: 0.1500 - val_loss: 2.9803 - learning_rate: 0.0010
Epoch 3/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1975 - loss: 2.9601 - val_accuracy: 0.2500 - val_loss: 2.9579 - learning_rate: 0.0010
Epoch 4/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.3086 - loss: 2.8967 - val_accuracy: 0.2500 - val_loss: 2.8904 - learning_rate: 0.0010
Epoch 5/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2840 - loss: 2.7294 - val_accuracy: 0.1500 - val_loss: 2.7135 - learning_rate: 0.0010
Epoch 6/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2778 - loss: 2.4328 - val_accuracy: 0.2500 - val_loss: 2.4522 - learning_rate: 0.0010
Epoch 7/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0